# Document Ingestion

This notebook does these steps:

1. Read the PDF files in the `documents` folder. Use OCR for scanned pages.
2. Divide the text into chunks.
3. Put the chunks into a Chroma Cloud collection.
4. Do a test of the retrieval. The retrieval gives only text.

**Code style:** The code follows the ASD-STE100 rules where they apply to code:

- One function does one task.
- Names use approved, simple words. A function name starts with a verb.
- One word has one meaning. "Chunk" is always a part of the text. "Page" is always a PDF page.
- Comments are short sentences in the active voice. An instruction comment uses the imperative.

## 1. Install the packages

Do this step one time only.

In [ ]:
%pip install -q chromadb pymupdf python-dotenv

## 2. Set the configuration

In [ ]:
import os
from pathlib import Path

import chromadb
import fitz
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from dotenv import load_dotenv

load_dotenv(Path(".env"))

DOCUMENTS_DIR = Path("documents")

# The collection uses the Chroma default embedding model (all-MiniLM-L6-v2).
# The model finds chunks with a similar meaning.
COLLECTION_NAME = "workshop_documents_default"

CHUNK_SIZE = 1000      # Maximum number of characters in one chunk.
CHUNK_OVERLAP = 200    # Number of characters that two adjacent chunks share.
BATCH_SIZE = 100       # Number of chunks in one write to Chroma.
TOP_K = 3              # Number of chunks that one query gives.

# OCR settings for documents that are not born digital (scanned pages).
# "auto":   use OCR only on a page that has too little text.
# "always": use OCR on all pages.
# "never":  do not use OCR.
OCR_MODE = "auto"
OCR_LANGUAGE = "eng"   # Tesseract language. Example: "eng+khm".
OCR_DPI = 300          # Resolution of the page image for OCR.
MIN_TEXT_LENGTH = 50   # A page with fewer characters than this gets OCR in "auto" mode.

## 3. Connect to Chroma Cloud

The `.env` file contains the connection values.
The collection uses the Chroma default embedding model. The first run downloads the model.

In [ ]:
def connect_to_chroma():
    """Connect to Chroma Cloud. Return the client."""
    return chromadb.CloudClient(
        tenant=os.environ["CHROMA_TENANT"],
        database=os.environ["CHROMA_DATABASE"],
        api_key=os.environ["CHROMA_API_KEY"],
    )


client = connect_to_chroma()
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=DefaultEmbeddingFunction(),
)
print(f"Collection: {collection.name}. Number of chunks: {collection.count()}.")

## 4. Read the PDF files

A born-digital PDF has a text layer. A scanned PDF has only images of the pages.
For a scanned page, the notebook uses OCR (Tesseract) to get the text.
Use `OCR_MODE` to control this.

To use OCR, install Tesseract one time. On macOS, use `brew install tesseract`.
For a Khmer document, also install the Khmer language data and set `OCR_LANGUAGE = "eng+khm"`.

In [ ]:
def clean_text(text):
    """Remove extra white space from the text."""
    return " ".join(text.split())


def read_ocr_text(page):
    """Get the text of the page with OCR."""
    textpage = page.get_textpage_ocr(language=OCR_LANGUAGE, dpi=OCR_DPI, full=True)
    return page.get_text(textpage=textpage)


def read_page_text(page, ocr_mode=OCR_MODE):
    """Get the text of one page. Use OCR as the OCR mode tells."""
    if ocr_mode == "always":
        return clean_text(read_ocr_text(page))
    text = clean_text(page.get_text())
    if ocr_mode == "auto" and len(text) < MIN_TEXT_LENGTH:
        text = clean_text(read_ocr_text(page))
    return text


def read_pages(pdf_path, ocr_mode=OCR_MODE):
    """Read a PDF file. Return a list of (page number, text) pairs."""
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = read_page_text(page, ocr_mode)
            if text:
                pages.append((page_number, text))
    return pages

### Test the OCR

This test makes a scanned copy of the first page of one PDF. The scanned copy has no text layer.
Then the test reads the scanned copy with and without OCR.

In [ ]:
def make_scanned_copy(pdf_path, page_index=0):
    """Make a one-page PDF that has only an image of the page."""
    with fitz.open(pdf_path) as pdf:
        image = pdf[page_index].get_pixmap(dpi=OCR_DPI)
    scanned = fitz.open()
    page = scanned.new_page(width=image.width, height=image.height)
    page.insert_image(page.rect, pixmap=image)
    return scanned


sample_path = sorted(DOCUMENTS_DIR.glob("*.pdf"))[0]
with make_scanned_copy(sample_path) as scanned:
    text_without_ocr = read_page_text(scanned[0], ocr_mode="never")
    text_with_ocr = read_page_text(scanned[0], ocr_mode="auto")

print(f"Characters without OCR: {len(text_without_ocr)}.")
print(f"Characters with OCR: {len(text_with_ocr)}.")
print(text_with_ocr[:300])

## 5. Divide the text into chunks

A chunk ends at a space. Thus, a chunk does not cut a word.

In [ ]:
def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Divide the text into chunks. Return the list of chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            space = text.rfind(" ", start, end)
            if space > start:
                end = space
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


def make_records(pdf_path):
    """Make the chunk records for one PDF file."""
    records = []
    for page_number, text in read_pages(pdf_path):
        for index, chunk in enumerate(split_text(text)):
            records.append({
                "id": f"{pdf_path.stem}-p{page_number}-c{index}",
                "text": chunk,
                "metadata": {"source": pdf_path.name, "page": page_number},
            })
    return records

## 6. Put the chunks into Chroma

The function uses `upsert`. Thus, you can run it again and it does not make duplicate chunks.

In [ ]:
def write_records(collection, records):
    """Write the records to the collection in batches."""
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        collection.upsert(
            ids=[record["id"] for record in batch],
            documents=[record["text"] for record in batch],
            metadatas=[record["metadata"] for record in batch],
        )


def ingest_documents(collection, documents_dir=DOCUMENTS_DIR):
    """Ingest all the PDF files in the folder."""
    for pdf_path in sorted(documents_dir.glob("*.pdf")):
        records = make_records(pdf_path)
        write_records(collection, records)
        print(f"{pdf_path.name}: {len(records)} chunks.")


ingest_documents(collection)
print(f"Total number of chunks: {collection.count()}.")

## 7. Retrieve the text

The function gives only the text of the chunks. It does not give the IDs, the metadata or the distances.

In [ ]:
def retrieve_text(collection, query, top_k=TOP_K):
    """Find the chunks that match the query. Return only their text."""
    result = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents"],
    )
    return result["documents"][0]

## 8. Do a test of the retrieval

In [ ]:
TEST_QUERIES = [
    "What are the objectives of the financial technology development policy?",
    "What are the goals of the e-services for business strategy?",
    "What are the principles of the IT architecture framework?",
]

for query in TEST_QUERIES:
    texts = retrieve_text(collection, query)
    assert texts, f"The query gave no text: {query}"
    assert all(isinstance(text, str) for text in texts), "The result must be text only."
    print(f"QUERY: {query}")
    for rank, text in enumerate(texts, start=1):
        print(f"  {rank}. {text[:300]}...")
    print()